In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/multiclassificationtask/sample_submission.csv
/kaggle/input/competitions/multiclassificationtask/train.csv
/kaggle/input/competitions/multiclassificationtask/test.csv


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
df = pd.read_csv('/kaggle/input/competitions/multiclassificationtask/train.csv')

print(df.shape)
print(df.head())
print(df.dtypes)

print(df.isnull().sum())
print(df['Status'].value_counts())
print(df['Edema'].unique())
print(df.groupby('Status')['Cholesterol'].apply(lambda x: x.isnull().mean()))
#df['Cholesterol'].fillna(df.groupby('Stage')['Cholesterol'].transform('median'))
df = df[df['Status'] != 'Y']
print(df['Status'].value_counts())

df['Age']=df['Age']  / 365
print(df['Age'].describe())
df[(df['Age'] <= 18) | (df['Age'] >=100)]
print(len(df[(df['Age'] < 18) | (df['Age'] > 100)]))
#Here's Step 1: Data Loading & Cleaning.

import numpy as np
import pandas as pd

# Load data
train = pd.read_csv('/kaggle/input/competitions/multiclassificationtask/train.csv')
test = pd.read_csv('/kaggle/input/competitions/multiclassificationtask/test.csv')

# Drop the bad 'Y' row
train = train[train['Status'] != 'Y'].copy()

# Convert Age from days to years
train['Age'] = train['Age'] / 365.25
test['Age'] = test['Age'] / 365.25

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nClass distribution:")
print(train['Status'].value_counts())
print("\nMissing values (train):")
print(train.isnull().sum()[train.isnull().sum() > 0])

In [ ]:
# Step 2: Feature Engineering — log transforms for skewed lab values + clinical ratio features:
def engineer_features(df):
    df = df.copy()
    
    # Log transform skewed numeric columns (safe for zeros/NaN)
    log_cols = ['Bilirubin', 'Cholesterol', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides']
    for col in log_cols:
        df[f'log_{col}'] = np.log1p(df[col])
    
    df['log_N_Days'] = np.log1p(df['N_Days'])
    
    # Clinical ratio features
    df['Bili_Albumin_ratio'] = df['Bilirubin'] / (df['Albumin'] + 1e-6)
    df['SGOT_AlkPhos_ratio'] = df['SGOT'] / (df['Alk_Phos'] + 1e-6)
    df['Stage_Bili'] = df['Stage'] * df['Bilirubin']
    df['Proth_Bili'] = df['Prothrombin'] * df['Bilirubin']
    
    return df

train = engineer_features(train)
test = engineer_features(test)

print("Train shape after feature engineering:", train.shape)
print("New columns:", [c for c in train.columns if c not in ['id','N_Days','Drug','Age','Sex',
      'Ascites','Hepatomegaly','Spiders','Edema','Bilirubin','Cholesterol','Albumin',
      'Copper','Alk_Phos','SGOT','Tryglicerides','Platelets','Prothrombin','Stage','Status']])

In [ ]:
#Step 3: Preprocessing Pipeline.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder

# Define feature columns
cat_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']
num_cols = [c for c in train.columns if c not in cat_cols + ['id', 'Status']]

# Preprocessing pipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Encode target: C→0, CL→1, D→2
le = LabelEncoder()
y = le.fit_transform(train['Status'])
X = train.drop(columns=['id', 'Status'])
X_test = test.drop(columns=['id'])

print("Classes:", le.classes_)  # should be ['C', 'CL', 'D']
print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("y distribution:", np.bincount(y))

In [ ]:
# Step 4: LightGBM with 5-Fold Cross-Validation:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

RANDOM_STATE = 42
N_SPLITS = 5

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

lgb_oof = np.zeros((len(X), 3))
lgb_test_preds = np.zeros((len(X_test), 3))

lgb_params = dict(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    verbose=-1
)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    # Fit preprocessor on train fold only (no leakage)
    X_tr_proc = preprocessor.fit_transform(X_tr)
    X_val_proc = preprocessor.transform(X_val)
    X_test_proc = preprocessor.transform(X_test)

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr_proc, y_tr,
        eval_set=[(X_val_proc, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]
    )

    lgb_oof[val_idx] = model.predict_proba(X_val_proc)
    lgb_test_preds += model.predict_proba(X_test_proc) / N_SPLITS

    fold_loss = log_loss(y_val, lgb_oof[val_idx])
    print(f"Fold {fold+1} log_loss: {fold_loss:.5f}")

lgb_cv_loss = log_loss(y, lgb_oof)
print(f"\nLightGBM CV log_loss: {lgb_cv_loss:.5f}")

In [ ]:
#Step 5: XGBoost Cross-Validation:
import xgboost as xgb

xgb_oof = np.zeros((len(X), 3))
xgb_test_preds = np.zeros((len(X_test), 3))

# Compute sample weights for class imbalance
class_counts = np.bincount(y)
sample_weights = np.array([1.0 / class_counts[yi] for yi in y])

xgb_params = dict(
    objective='multi:softprob',
    num_class=3,
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    verbosity=0
)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    w_tr = sample_weights[train_idx]

    X_tr_proc = preprocessor.fit_transform(X_tr)
    X_val_proc = preprocessor.transform(X_val)
    X_test_proc = preprocessor.transform(X_test)

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(
        X_tr_proc, y_tr,
        sample_weight=w_tr,
        eval_set=[(X_val_proc, y_val)],
        verbose=200
    )

    xgb_oof[val_idx] = model.predict_proba(X_val_proc)
    xgb_test_preds += model.predict_proba(X_test_proc) / N_SPLITS

    fold_loss = log_loss(y_val, xgb_oof[val_idx])
    print(f"Fold {fold+1} log_loss: {fold_loss:.5f}")

xgb_cv_loss = log_loss(y, xgb_oof)
print(f"\nXGBoost CV log_loss: {xgb_cv_loss:.5f}")
xgb_oof = np.zeros((len(X), 3))
xgb_test_preds = np.zeros((len(X_test), 3))

# Softer sqrt weighting instead of full inverse frequency
class_counts = np.bincount(y)
sample_weights = np.array([1.0 / np.sqrt(class_counts[yi]) for yi in y])
sample_weights = sample_weights / sample_weights.mean()  # normalize

xgb_params = dict(
    objective='multi:softprob',
    num_class=3,
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    verbosity=0
)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    w_tr = sample_weights[train_idx]

    X_tr_proc = preprocessor.fit_transform(X_tr)
    X_val_proc = preprocessor.transform(X_val)
    X_test_proc = preprocessor.transform(X_test)

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(
        X_tr_proc, y_tr,
        sample_weight=w_tr,
        eval_set=[(X_val_proc, y_val)],
        verbose=200
    )

    xgb_oof[val_idx] = model.predict_proba(X_val_proc)
    xgb_test_preds += model.predict_proba(X_test_proc) / N_SPLITS

    fold_loss = log_loss(y_val, xgb_oof[val_idx])
    print(f"Fold {fold+1} log_loss: {fold_loss:.5f}")

xgb_cv_loss = log_loss(y, xgb_oof)
print(f"\nXGBoost CV log_loss: {xgb_cv_loss:.5f}")

In [ ]:
# Step 6: Ensemble + Submission:
# Normalize probabilities to sum to 1 (fixes the warning)
def normalize_probs(probs):
    return probs / probs.sum(axis=1, keepdims=True)

lgb_oof_norm = normalize_probs(lgb_oof)
xgb_oof_norm = normalize_probs(xgb_oof)
lgb_test_norm = normalize_probs(lgb_test_preds)
xgb_test_norm = normalize_probs(xgb_test_preds)

# Try different ensemble weights
for lgb_w in [0.3, 0.4, 0.5]:
    xgb_w = 1 - lgb_w
    ensemble_oof = lgb_w * lgb_oof_norm + xgb_w * xgb_oof_norm
    loss = log_loss(y, ensemble_oof)
    print(f"LGB {lgb_w:.1f} + XGB {xgb_w:.1f} → log_loss: {loss:.5f}")

# Pick best weight (likely xgb-heavy since xgb scored better)
best_lgb_w = 0.3
best_xgb_w = 0.7
ensemble_test = best_lgb_w * lgb_test_norm + best_xgb_w * xgb_test_norm
ensemble_test = normalize_probs(ensemble_test)

# Build submission
submission = pd.DataFrame({
    'id': test['id'],
    'Status_C': ensemble_test[:, 0],
    'Status_CL': ensemble_test[:, 1],
    'Status_D': ensemble_test[:, 2],
})

submission.to_csv('submission.csv', index=False)
print("\nSubmission saved!")
print(submission.head())
print("\nProb sums (should all be 1.0):", ensemble_test.sum(axis=1)[:5])
# Check pure XGB and lighter LGB blends
for lgb_w in [0.0, 0.1, 0.2]:
    xgb_w = 1 - lgb_w
    ensemble_oof = lgb_w * lgb_oof_norm + xgb_w * xgb_oof_norm
    loss = log_loss(y, ensemble_oof)
    print(f"LGB {lgb_w:.1f} + XGB {xgb_w:.1f} → log_loss: {loss:.5f}")
best_test = normalize_probs(xgb_test_norm)

submission = pd.DataFrame({
    'id': test['id'],
    'Status_C': best_test[:, 0],
    'Status_CL': best_test[:, 1],
    'Status_D': best_test[:, 2],
})
submission.to_csv('submission.csv', index=False)
print("Saved! CV log_loss:", log_loss(y, normalize_probs(xgb_oof_norm)))
print(submission.head())

/kaggle/input/competitions/multiclassificationtask/sample_submission.csv
/kaggle/input/competitions/multiclassificationtask/train.csv
/kaggle/input/competitions/multiclassificationtask/test.csv
(15000, 20)
   id  N_Days             Drug      Age Sex Ascites Hepatomegaly Spiders  \
0   0  2178.0  D-penicillamine  16374.0   F       N            N       N   
1   1  2644.0  D-penicillamine  17774.0   F       N            N       N   
2   2  3069.0          Placebo  17844.0   F       N            N       N   
3   3  2216.0          Placebo  19221.0   F       N            Y       Y   
4   4  2256.0          Placebo  21600.0   F       N            N       N   

  Edema  Bilirubin  Cholesterol  Albumin  Copper  Alk_Phos    SGOT  \
0     N        0.5        263.0     3.20    43.0    1110.0  106.95   
1     N        0.8        280.0     3.60    22.0     678.0   62.00   
2     N        1.1        408.0     4.40    54.0    2108.0  142.60   
3     N        0.8        252.0     3.70    36.0     843.

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 1 log_loss: 0.43168
[200]	valid_0's multi_logloss: 0.401612


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 2 log_loss: 0.40152
[200]	valid_0's multi_logloss: 0.41604


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


[200]	validation_0-mlogloss:0.39338
[359]	validation_0-mlogloss:0.39143
Fold 4 log_loss: 0.39095
[0]	validation_0-mlogloss:0.76340


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


[200]	validation_0-mlogloss:0.39531
[400]	validation_0-mlogloss:0.38634
[407]	validation_0-mlogloss:0.38649
Fold 5 log_loss: 0.38545

XGBoost CV log_loss: 0.39214
LGB 0.3 + XGB 0.7 → log_loss: 0.39372
LGB 0.4 + XGB 0.6 → log_loss: 0.39499
LGB 0.5 + XGB 0.5 → log_loss: 0.39664

Submission saved!
      id  Status_C  Status_CL  Status_D
0  15000  0.348529   0.005560  0.645911
1  15001  0.746313   0.025241  0.228447
2  15002  0.610423   0.005426  0.384150
3  15003  0.989388   0.000399  0.010213
4  15004  0.147842   0.205869  0.646289

Prob sums (should all be 1.0): [1. 1. 1. 1. 1.]
LGB 0.0 + XGB 1.0 → log_loss: 0.39214
LGB 0.1 + XGB 0.9 → log_loss: 0.39228
LGB 0.2 + XGB 0.8 → log_loss: 0.39281
Saved! CV log_loss: 0.3921376403044021
      id  Status_C  Status_CL  Status_D
0  15000  0.349144   0.005878  0.644979
1  15001  0.752772   0.027735  0.219493
2  15002  0.616709   0.006296  0.376995
3  15003  0.989585   0.000392  0.010023
4  15004  0.159745   0.198382  0.641873


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [3]:
def engineer_features(df):
    df = df.copy()
    
    log_cols = ['Bilirubin', 'Cholesterol', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides']
    for col in log_cols:
        df[f'log_{col}'] = np.log1p(df[col])
    
    df['log_N_Days'] = np.log1p(df['N_Days'])
    
    # Original ratios
    df['Bili_Albumin_ratio'] = df['Bilirubin'] / (df['Albumin'] + 1e-6)
    df['SGOT_AlkPhos_ratio'] = df['SGOT'] / (df['Alk_Phos'] + 1e-6)
    df['Stage_Bili'] = df['Stage'] * df['Bilirubin']
    df['Proth_Bili'] = df['Prothrombin'] * df['Bilirubin']
    
    # New features from analysis
    df['Liver_Score_1'] = (df['Bilirubin'] * 0.5) + (df['Cholesterol'] * 0.003) - (df['Albumin'] * 0.5)
    df['Liver_Score_2'] = np.log1p(df['Copper']) + np.log1p(df['Alk_Phos']) + np.log1p(df['SGOT'])
    df['Platelets_Prothrombin'] = df['Platelets'] * df['Prothrombin']
    
    return df

# Re-apply to both
train = engineer_features(train)
test = engineer_features(test)

# Rebuild X, X_test
X = train.drop(columns=['id', 'Status'])
X_test = test.drop(columns=['id'])
print("New shape:", X.shape)  # should be (14999, 32)

New shape: (14999, 32)


In [5]:
cat_oof = np.zeros((len(X), 3))
cat_test_preds = np.zeros((len(X_test), 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    num_imputer = SimpleImputer(strategy='median')
    cat_imputer = SimpleImputer(strategy='most_frequent')

    X_tr_num = X_tr[num_cols].copy(); X_val_num = X_val[num_cols].copy(); X_test_num = X_test[num_cols].copy()
    X_tr_num[:] = num_imputer.fit_transform(X_tr_num)
    X_val_num[:] = num_imputer.transform(X_val_num)
    X_test_num[:] = num_imputer.transform(X_test_num)

    X_tr_cat = X_tr[cat_cols].copy(); X_val_cat = X_val[cat_cols].copy(); X_test_cat = X_test[cat_cols].copy()
    X_tr_cat[:] = cat_imputer.fit_transform(X_tr_cat)
    X_val_cat[:] = cat_imputer.transform(X_val_cat)
    X_test_cat[:] = cat_imputer.transform(X_test_cat)

    X_tr_full = pd.concat([X_tr_num.reset_index(drop=True), X_tr_cat.reset_index(drop=True)], axis=1)
    X_val_full = pd.concat([X_val_num.reset_index(drop=True), X_val_cat.reset_index(drop=True)], axis=1)
    X_test_full = pd.concat([X_test_num.reset_index(drop=True), X_test_cat.reset_index(drop=True)], axis=1)
    cat_indices = [X_tr_full.columns.get_loc(c) for c in cat_cols]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3.0,
        loss_function='MultiClass',
        eval_metric='MultiClass',
        random_seed=RANDOM_STATE,
        verbose=200,
        early_stopping_rounds=50,
        # No class_weights this time
    )

    model.fit(X_tr_full, y_tr, cat_features=cat_indices, eval_set=(X_val_full, y_val))
    cat_oof[val_idx] = model.predict_proba(X_val_full)
    cat_test_preds += model.predict_proba(X_test_full) / N_SPLITS

    fold_loss = log_loss(y_val, cat_oof[val_idx])
    print(f"Fold {fold+1} log_loss: {fold_loss:.5f}")

cat_cv_loss = log_loss(y, cat_oof)
print(f"\nCatBoost CV log_loss: {cat_cv_loss:.5f}")

0:	learn: 1.0396195	test: 1.0402658	best: 1.0402658 (0)	total: 28.7ms	remaining: 28.6s
200:	learn: 0.3667475	test: 0.4140894	best: 0.4140894 (200)	total: 4.34s	remaining: 17.2s
400:	learn: 0.3318738	test: 0.4041043	best: 0.4041043 (400)	total: 8.73s	remaining: 13s
600:	learn: 0.3067297	test: 0.3995843	best: 0.3995635 (598)	total: 13s	remaining: 8.66s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.3986009388
bestIteration = 661

Shrink model to first 662 iterations.
Fold 1 log_loss: 0.39860
0:	learn: 1.0402945	test: 1.0401512	best: 1.0401512 (0)	total: 26.8ms	remaining: 26.8s
200:	learn: 0.3739194	test: 0.3929006	best: 0.3928155 (197)	total: 4.34s	remaining: 17.2s
400:	learn: 0.3379738	test: 0.3829088	best: 0.3828194 (399)	total: 8.7s	remaining: 13s
600:	learn: 0.3106909	test: 0.3806147	best: 0.3805272 (569)	total: 13.1s	remaining: 8.71s
800:	learn: 0.2895781	test: 0.3793314	best: 0.3792815 (795)	total: 17.5s	remaining: 4.34s
Stopped by overfitting detector  (50 ite

In [6]:
cat_oof_norm = normalize_probs(cat_oof)
cat_test_norm = normalize_probs(cat_test_preds)

# Search best XGB + CatBoost blend
print("XGB + CatBoost blends:")
for cat_w in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    xgb_w = 1 - cat_w
    ensemble = xgb_w * xgb_oof_norm + cat_w * cat_oof_norm
    loss = log_loss(y, ensemble)
    print(f"XGB {xgb_w:.1f} + CAT {cat_w:.1f} → log_loss: {loss:.5f}")

XGB + CatBoost blends:
XGB 0.5 + CAT 0.5 → log_loss: 0.38438
XGB 0.4 + CAT 0.6 → log_loss: 0.38432
XGB 0.3 + CAT 0.7 → log_loss: 0.38471
XGB 0.2 + CAT 0.8 → log_loss: 0.38557
XGB 0.1 + CAT 0.9 → log_loss: 0.38696
XGB 0.0 + CAT 1.0 → log_loss: 0.38898


In [7]:
best_test = normalize_probs(0.4 * xgb_test_norm + 0.6 * cat_test_norm)

submission = pd.DataFrame({
    'id': test['id'],
    'Status_C': best_test[:, 0],
    'Status_CL': best_test[:, 1],
    'Status_D': best_test[:, 2],
})
submission.to_csv('submission.csv', index=False)

print("Final CV log_loss: 0.38432")
print("Saved submission.csv")
print(submission.head())
print("\nProb sums:", best_test.sum(axis=1)[:5])

Final CV log_loss: 0.38432
Saved submission.csv
      id  Status_C  Status_CL  Status_D
0  15000  0.368149   0.005179  0.626672
1  15001  0.813275   0.018199  0.168526
2  15002  0.661095   0.007626  0.331279
3  15003  0.986095   0.000821  0.013084
4  15004  0.238113   0.157994  0.603892

Prob sums: [1. 1. 1. 1. 1.]


In [8]:
# Try XGB + CAT + LGB triple blend
print("Triple blend search:")
best_loss = 1.0
best_weights = None

for xgb_w in [0.2, 0.3, 0.4]:
    for cat_w in [0.4, 0.5, 0.6]:
        lgb_w = round(1 - xgb_w - cat_w, 1)
        if lgb_w < 0:
            continue
        ensemble = xgb_w * xgb_oof_norm + cat_w * cat_oof_norm + lgb_w * lgb_oof_norm
        loss = log_loss(y, ensemble)
        print(f"XGB {xgb_w} + CAT {cat_w} + LGB {lgb_w} → {loss:.5f}")
        if loss < best_loss:
            best_loss = loss
            best_weights = (xgb_w, cat_w, lgb_w)

print(f"\nBest: XGB {best_weights[0]} + CAT {best_weights[1]} + LGB {best_weights[2]} → {best_loss:.5f}")

Triple blend search:
XGB 0.2 + CAT 0.4 + LGB 0.4 → 0.38715
XGB 0.2 + CAT 0.5 + LGB 0.3 → 0.38549
XGB 0.2 + CAT 0.6 + LGB 0.2 → 0.38466
XGB 0.3 + CAT 0.4 + LGB 0.3 → 0.38606
XGB 0.3 + CAT 0.5 + LGB 0.2 → 0.38476
XGB 0.3 + CAT 0.6 + LGB 0.1 → 0.38429
XGB 0.4 + CAT 0.4 + LGB 0.2 → 0.38531
XGB 0.4 + CAT 0.5 + LGB 0.1 → 0.38438
XGB 0.4 + CAT 0.6 + LGB 0.0 → 0.38432

Best: XGB 0.3 + CAT 0.6 + LGB 0.1 → 0.38429
